##### Overview of Supervised Fine-tuning (SFT)

Given a prompt + Response, we want to fine-tune a Language Model on Specific task say predicting the next token.

#### Installing Packages

In [1]:
!pip -q install datasets

#### Importing and initialling model GPT-2

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import torch


In [3]:
model_name="gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

##### Testing tokenizer

##### Encoder

In [4]:
text = "Hello how are you"
tokens = tokenizer.encode(text)
print(tokens)

[15496, 703, 389, 345]


###### Decoding

In [5]:
print(tokenizer.decode(tokens))

Hello how are you


##### Using custom dataset

In [6]:
# loading the sentiment dataset from hf
dataset_name = "sst2"
ds = load_dataset(dataset_name)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [7]:
ds

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

##### Spliting Data

In [8]:
ds_train, ds_val = ds['train'], ds['validation']
print(ds_train)
print(ds_val)



Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 67349
})
Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 872
})


In [9]:
# Displaying some samples
display(ds_train[:10])

{'idx': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 'sentence': ['hide new secretions from the parental units ',
  'contains no wit , only labored gags ',
  'that loves its characters and communicates something rather beautiful about human nature ',
  'remains utterly satisfied to remain the same throughout ',
  'on the worst revenge-of-the-nerds clichés the filmmakers could dredge up ',
  "that 's far too tragic to merit such superficial treatment ",
  'demonstrates that the director of such hollywood blockbusters as patriot games can still turn out a small , personal film with an emotional wallop . ',
  'of saucy ',
  "a depressed fifteen-year-old 's suicidal poetry ",
  "are more deeply thought through than in most ` right-thinking ' films "],
 'label': [0, 0, 1, 0, 0, 0, 1, 1, 0, 1]}

##### Tokenizing a Dataset

In [10]:
# Tokenizing a dataset
def tokenize_ds(batch):
  return tokenizer(batch['sentence'])

# Avoiding for lop we map each sentence with the label
map_kwargs = {
    'batched':True,
    'batch_size':512,
    'remove_columns':['idx', 'sentence', 'label']
}

# Applying tokenizing function
tokenized_dataset_train = ds_train.map(tokenize_ds, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize_ds, **map_kwargs)


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

In [11]:
tokenized_dataset_train[0]

{'input_ids': [24717, 649, 3200, 507, 422, 262, 21694, 4991, 220],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [12]:
## Decoding we have
for i , seq in enumerate(tokenized_dataset_train[5:20]['input_ids']):
  print(f"{i + 1}: {tokenizer.decode(seq)}")

1: that 's far too tragic to merit such superficial treatment 
2: demonstrates that the director of such hollywood blockbusters as patriot games can still turn out a small , personal film with an emotional wallop . 
3: of saucy 
4: a depressed fifteen-year-old 's suicidal poetry 
5: are more deeply thought through than in most ` right-thinking ' films 
6: goes to absurd lengths 
7: for those moviegoers who complain that ` they do n't make movies like they used to anymore 
8: the part where nothing 's happening , 
9: saw how bad this movie was 
10: lend some dignity to a dumb story 
11: the greatest musicians 
12: cold movie 
13: with his usual intelligence and subtlety 
14: redundant concept 
15: swimming is above all about a young woman 's face , and by casting an actress whose face projects that woman 's doubts and yearnings , it succeeds . 


In [13]:
# Filtering dataset for short sentences
tokenized_dataset_train = tokenized_dataset_train.filter(lambda x: len(x['input_ids']) > 5, batched=False)
tokenized_dataset_val = tokenized_dataset_val.filter(lambda x: len(x['input_ids']) > 5, batched=False)

print(len(tokenized_dataset_train), len(tokenized_dataset_val))

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

Filter:   0%|          | 0/872 [00:00<?, ? examples/s]

49401 867


##### Training

###### Prapering DataLoader

In [14]:

tokenized_dataset_train.set_format(type='torch')
tokenized_dataset_val.set_format(type='torch')

In [15]:
tokenized_dataset_train[:5]

{'input_ids': [tensor([24717,   649,  3200,   507,   422,   262, 21694,  4991,   220]),
  tensor([ 3642,  1299,   645, 20868,   837,   691,  2248,  1850,   308,  3775,
            220]),
  tensor([ 5562, 10408,   663,  3435,   290, 48556,  1223,  2138,  4950,   546,
           1692,  3450,   220]),
  tensor([ 2787,  1299, 15950, 11378,   284,  3520,   262,   976,  3690,   220]),
  tensor([  261,   262,  5290, 15827,    12,  1659,    12,  1169,    12,  1008,
           9310, 35478, 20954,   262, 28303,   714, 47478,   469,   510,   220])],
 'attention_mask': [tensor([1, 1, 1, 1, 1, 1, 1, 1, 1]),
  tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
  tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
  tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
  tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])]}

In [16]:
# Adding padding
tokenizer.pad_token = tokenizer.eos_token

In [17]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling


data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
dataloader_params = {
    'batch_size':32,
    'collate_fn':data_collator
}

train_dataloader = DataLoader(tokenized_dataset_train, **dataloader_params)
val_dataloader = DataLoader(tokenized_dataset_val, **dataloader_params)

In [18]:
len(train_dataloader)

1544

In [19]:
batch = next(iter(train_dataloader))
print(batch.keys())
print(batch['input_ids'].shape)

KeysView({'input_ids': tensor([[24717,   649,  3200,  ..., 50256, 50256, 50256],
        [ 3642,  1299,   645,  ..., 50256, 50256, 50256],
        [ 5562, 10408,   663,  ..., 50256, 50256, 50256],
        ...,
        [  672, 35260,   284,  ..., 50256, 50256, 50256],
        [ 1169,  2104,   966,  ..., 50256, 50256, 50256],
        [  292,   484,  1282,  ..., 50256, 50256, 50256]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([[24717,   649,  3200,  ...,  -100,  -100,  -100],
        [ 3642,  1299,   645,  ...,  -100,  -100,  -100],
        [ 5562, 10408,   663,  ...,  -100,  -100,  -100],
        ...,
        [  672, 35260,   284,  ...,  -100,  -100,  -100],
        [ 1169,  2104,   966,  ...,  -100,  -100,  -100],
        [  292,   484,  1282,  ...,  -100,  -100,  -100]])})
tor

##### Perform SFT

In [20]:
import torch

In [21]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 10

##### Training loop

In [22]:
# Validating using our validation test set
def validate(epoch):
  model.eval()
  total_loss = 0.0
  for i, batch in enumerate(val_dataloader):
    batch = batch.to(device)
    with torch.no_grad():
      outputs = model(**batch)
      loss = outputs.loss
      total_loss += loss.item()
  print(f"Validation loss after epoch {epoch + 1}: {total_loss / (i + 1)}")

In [23]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
validate(0)
for epoch in range(num_epochs):
  model.train()
  for batch in train_dataloader:
    batch = batch.to(device)
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
  validate(epoch+1)
  print(f"Epoch {epoch + 1} completed")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Validation loss after epoch 1: 5.18176177569798
Validation loss after epoch 2: 3.9842155235154286
Epoch 1 completed


KeyboardInterrupt: 

In [24]:
model.save_pretrained('./sft_model')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [25]:
!zip -r sft_model.zip sft_model_1/

	zip warning: name not matched: sft_model_1/

zip error: Nothing to do! (try: zip -r sft_model.zip . -i sft_model_1/)


In [ ]:
model = AutoModelForCausalLM.from_pretrained('./sft_model')
model

#### Saving model to drive

In [26]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive Mounted Successful!")

Mounted at /content/drive
Drive Mounted Successful!


In [27]:
# Zipping Model Folder to Zip file
!zip -r sft_model.zip sft_model/

  adding: sft_model/ (stored 0%)
  adding: sft_model/model.safetensors (deflated 7%)
  adding: sft_model/config.json (deflated 52%)
  adding: sft_model/generation_config.json (deflated 25%)


In [33]:
import shutil

In [36]:
# Saving model to drive
#!cp -r sft_model.zip "/content/drive/MyDrive/AI Safety/Lab/Model"

shutil.copy('/content/sft_model.zip', '/content/drive/MyDrive/AI Safety /Lab/Model')
print("Model Copied to Drive")

Model Copied to Drive


#### Training Reward Model

```
# This is formatted as code
```

